# Week 3 — Retrieval benchmark (dense vs BM25 vs hybrid)

Fixed query set: `data/benchmarks/fiqa_sample_query_set.json`.

**Prerequisites:** `python src/scripts/ingest.py` (Qdrant + BM25).

**Outputs**
- `docs/reports/week3_retrieval_comparison.md` — **latency** (warm-up + repeats, median / p95) + BM25 overlap microbenchmark
- `docs/reports/week3_retrieval_comparison_exploratory_snapshot.md` — single-pass overlap + snippets
- `data/benchmarks/fiqa_sample_benchmark_timings.csv` — raw timed samples
- `data/benchmarks/fiqa_sample_benchmark_runs.csv` — single-pass quality rows

In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

load_dotenv(REPO_ROOT / ".env")

from app.config import get_settings
from app.evaluation.retrieval_benchmark import (
    BenchmarkTimingConfig,
    benchmark_query_set,
    benchmark_query_set_timed,
    load_query_set,
    render_exploratory_report,
    render_latency_report,
    results_to_records,
    summarize_timed_latencies,
    timed_results_to_records,
)
from app.retrieval import Bm25ChunkIndex, QdrantChunkStore, create_embedding_provider

settings = get_settings()
if settings.hf_hub_disable_symlinks_warning:
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

QUERY_SET_PATH = REPO_ROOT / "data" / "benchmarks" / "fiqa_sample_query_set.json"
LATENCY_REPORT_PATH = REPO_ROOT / "docs" / "reports" / "week3_retrieval_comparison.md"
EXPLORATORY_REPORT_PATH = (
    REPO_ROOT / "docs" / "reports" / "week3_retrieval_comparison_exploratory_snapshot.md"
)
TIMINGS_CSV_PATH = REPO_ROOT / "data" / "benchmarks" / "fiqa_sample_benchmark_timings.csv"
QUALITY_CSV_PATH = REPO_ROOT / "data" / "benchmarks" / "fiqa_sample_benchmark_runs.csv"

TIMING_CONFIG = BenchmarkTimingConfig(
    warmup_repeats=3,
    timed_repeats_per_query=7,
)

spec = load_query_set(QUERY_SET_PATH)
print(f"Loaded {len(spec.queries)} queries (top_k={spec.top_k})")

Loaded 10 queries (top_k=5)


In [2]:
bm25_path = settings.resolved_bm25_index_path(spec.collection)
qdrant_path = settings.resolved_qdrant_path()

if not qdrant_path or not Path(qdrant_path).exists():
    raise FileNotFoundError(f"Qdrant missing: {qdrant_path}")
if not bm25_path.is_file():
    raise FileNotFoundError(f"BM25 missing: {bm25_path}")

load_start = time.perf_counter()
embedder = create_embedding_provider(
    settings.embedding_method,
    model_name=settings.embedding_model,
    cohere_api_key=settings.cohere_api_key,
    cohere_embed_model=settings.cohere_embed_model,
    cohere_embed_batch_size=settings.cohere_embed_batch_size,
    cohere_embed_batch_sleep_seconds=settings.cohere_embed_batch_sleep_seconds,
)
model_load_ms = (time.perf_counter() - load_start) * 1000.0

bm25_index = Bm25ChunkIndex.load(bm25_path)
store = QdrantChunkStore.from_path(qdrant_path, spec.collection)
if not store.collection_exists():
    store.close()
    raise RuntimeError(f"Collection {spec.collection} not found")

print(f"Model load (excluded from timed runs): {model_load_ms:.1f} ms")
print(f"Chunks indexed: {bm25_index.num_chunks}, docs: {bm25_index.unique_doc_count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model load (excluded from timed runs): 15724.6 ms
Chunks indexed: 1900, docs: 999


In [3]:
try:
    timed = benchmark_query_set_timed(
        spec,
        store=store,
        embedder=embedder,
        bm25_index=bm25_index,
        config=TIMING_CONFIG,
        model_load_ms=model_load_ms,
    )
    exploratory = timed.quality_results
finally:
    store.close()

display(pd.DataFrame(summarize_timed_latencies(timed)).T)
display(pd.DataFrame(timed_results_to_records(timed)).head(12))

,p50_ms,p95_ms,mean_ms,min_ms,max_ms,n
dense,18.46715,21.306960,18.687631,16.1285,21.9932,70.0
bm25,23.39165,27.739245,23.372030,18.4636,28.5507,70.0
hybrid,41.91615,46.448205,42.072269,36.5362,48.5224,70.0


,query_id,theme,method,repeat,latency_ms
0,q01,brokerage,dense,1,18.5055
1,q01,brokerage,dense,2,20.3440
2,q01,brokerage,dense,3,18.4286
3,q01,brokerage,dense,4,16.9378
4,q01,brokerage,dense,5,17.6430
5,q01,brokerage,dense,6,17.0018
6,q01,brokerage,dense,7,18.0915
7,q01,brokerage,bm25,1,20.9982
8,q01,brokerage,bm25,2,25.1924
9,q01,brokerage,bm25,3,23.0346


In [ ]:
latency_md = render_latency_report(
    timed,
    embedding_method=settings.embedding_method,
    embedding_model=settings.embedding_model,
)
exploratory_md = render_exploratory_report(
    spec,
    exploratory,
    embedding_method=settings.embedding_method,
    embedding_model=settings.embedding_model,
)

LATENCY_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
LATENCY_REPORT_PATH.write_text(latency_md, encoding="utf-8")
EXPLORATORY_REPORT_PATH.write_text(exploratory_md, encoding="utf-8")
TIMINGS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(timed_results_to_records(timed)).to_csv(TIMINGS_CSV_PATH, index=False)
pd.DataFrame(results_to_records(exploratory)).to_csv(QUALITY_CSV_PATH, index=False)

print(f"Wrote {LATENCY_REPORT_PATH}")
print(f"Wrote {EXPLORATORY_REPORT_PATH}")

Wrote D:\Repos\RAG\RAG\docs\reports\week3_retrieval_comparison.md
Wrote D:\Repos\RAG\RAG\docs\reports\week3_retrieval_comparison_exploratory_snapshot.md


In [5]:
pd.DataFrame(timed_results_to_records(timed))

,query_id,theme,method,repeat,latency_ms
0,q01,brokerage,dense,1,18.5055
1,q01,brokerage,dense,2,20.3440
2,q01,brokerage,dense,3,18.4286
3,q01,brokerage,dense,4,16.9378
4,q01,brokerage,dense,5,17.6430
...,...,...,...,...,...
205,q10,401k,hybrid,3,39.7657
206,q10,401k,hybrid,4,37.7629
207,q10,401k,hybrid,5,44.6597
208,q10,401k,hybrid,6,44.5748


In [6]:
pd.DataFrame(results_to_records(exploratory))

,query_id,theme,method,latency_ms,num_hits,top_doc_ids,top_keys
0,q01,brokerage,dense,17.8485,5,"4883,4883,4883,4883,7411","4883:0,4883:1,4883:2,4883:3,7411:0"
1,q01,brokerage,bm25,24.7509,5,"4883,4883,4883,4883,4883","4883:0,4883:1,4883:2,4883:3,4883:4"
2,q01,brokerage,hybrid,43.8762,5,"4883,4883,4883,4883,4883","4883:0,4883:1,4883:2,4883:3,4883:4"
3,q02,credit,dense,16.3596,5,"5747,8496,5747,9814,2768","2768:0,5747:2,5747:3,8496:0,9814:3"
4,q02,credit,bm25,20.4456,5,"6341,9814,8950,6343,991","6341:0,6343:0,8950:0,9814:0,991:0"
5,q02,credit,hybrid,38.6856,5,"5747,6341,8496,9814,5747","5747:2,5747:3,6341:0,8496:0,9814:0"
6,q03,investing,dense,17.8417,5,"5188,8266,8000,5188,9656","5188:0,5188:4,8000:0,8266:0,9656:0"
7,q03,investing,bm25,25.1873,5,"9656,6339,680,8974,5188","5188:0,6339:0,680:1,8974:3,9656:0"
8,q03,investing,hybrid,44.4448,5,"5188,9656,680,8974,680","5188:0,680:0,680:1,8974:3,9656:0"
9,q04,taxes,dense,18.3616,5,"7423,7423,5224,6891,5180","5180:3,5224:0,6891:0,7423:0,7423:1"
